# T9 It3: the decision on altered photos (red team with real pipeline runs)

60 official photos (both halves, all height bands) and 60 NUDT 707 photos, each run through the unchanged pipeline seven times with an altered
input (`Rich_Dump_Perturbed`, block `train/checks/perturb.py`): thermal polarity flipped, blur, noise, half resolution, tone curve, heading
+15 deg, height +20 %. The clean runs of the same photos are the T6 / T8 dumps. Rules compared (all judges trained on NUDT 101-606 only):
- today's rules without agreement (PC bar 8 / 11; Jetson bar 8, cap 10, M = 5);
- the recommended judge (cut 0.9 PC / 0.925 Jetson, 12-inlier override), Jetson with early exit;
- the judge **without the 12-inlier override** (It2: same score on clean data, far fewer wrong answers when matches inflate), and at cut 0.98.
The safety question is the wrong answers; the usefulness question is the score drop against the clean run.

T10 updates: Jetson rules use context from the 5 matched candidates; T10 variants V3 (scale augmentation) and V5 (relative inliers + augmentation) added.

In [ ]:
import os, sys, time
os.environ.setdefault("OMP_NUM_THREADS", "8")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
from train.checks.decision_signals import build_signal_table
from train.checks.holdout import NUDT_DESIGN, load_t8_table
from train.checks.perturb import KINDS
RES = ROOT / "tests/Test_Iteration_9/results"
(RES / "signals").mkdir(exist_ok=True)
t0 = time.time()
for base in ["official", "nudt707"]:
    for kind in KINDS:
        out = RES / "signals" / f"{base}_{kind}.csv.gz"
        if not out.exists():
            build_signal_table(RES / "rich_dump" / f"{base}_{kind}", f"{base}_{kind}", int(os.environ.get("WORKERS", "12"))).to_csv(out, index=False)
print("signal tables %.0f s" % (time.time() - t0))

In [ ]:
import gc
DROP = ["valid_fraction", "inl_lo", "lnfa_t3", "lnfa_s3", "t_shift_px", "t_dx", "t_dy", "t_err_sigma", "tx", "ty", "guided15", "inl_t2", "inl_t3_15"]
def slim(df):
    """Memory: drop signals no rule here uses; float64 -> float32 (14 GB laptop, a GPU dump running alongside)."""
    df = df.drop(columns=[k for k in DROP if k in df.columns])
    f = df.select_dtypes("float64").columns
    df[f] = df[f].astype("float32")
    return df
ref = load_t8_table(ROOT, locked=False, blind=True)
ref.loc[ref.data.str.startswith("official"), "data"] = "official"
ref = slim(ref[ref.data.isin(NUDT_DESIGN + ["official", "nudt707"])])
ids = {"official": set(pd.read_csv(RES / "photos_official.csv").photo), "nudt707": set(pd.read_csv(RES / "photos_nudt707.csv").photo)}
clean = pd.concat([ref[(ref.data == b) & ref.photo.isin(ids[b])].assign(data=f"{b}_clean") for b in ids])
alt = pd.concat([pd.read_csv(RES / "signals" / f"{b}_{k}.csv.gz") for b in ids for k in KINDS])
alt = slim(cj.add_context(cj.add_features(alt)))
design = ref[ref.data.isin(NUDT_DESIGN)].reset_index(drop=True)
T = pd.concat([clean, alt], ignore_index=True)
P = cj.judge_probability(pd.concat([design, T], ignore_index=True), NUDT_DESIGN, cj.FEATURES_CTX, "gbm")[0][len(design):]   # trained on NUDT 101-606 only
T5 = cj.add_context(T, m=5)                       # Jetson: context from the 5 matched candidates (T10 fix 1a)
design5 = cj.add_context(design, m=5)
P5 = cj.judge_probability(pd.concat([design5, T5], ignore_index=True), NUDT_DESIGN, cj.FEATURES_CTX, "gbm")[0][len(design5):]
print(T.groupby("data").photo.nunique().to_string())
del ref; gc.collect()

In [ ]:
# T10 part 2 variants (train/checks/scale_robust.py), trained on the NUDT design sets only:
# V3 scale augmentation (x0.5 ... x2 copies), V5 relative inliers (vs the run's far-candidate level) + augmentation. No override, LTT cut.
import train.checks.scale_robust as sr
AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]
def variant(feats, inc, rel, ctx_m):
    tr_t = design if ctx_m is None else cj.add_context(design, m=ctx_m)
    lv = sr.far_levels(tr_t, ctx_m); prior = dict(far_inl=lv.far_inl.mean(), far_n22=lv.far_n22.mean())
    model, p_oof, st = sr.fit_judge(tr_t, NUDT_DESIGN, feats, inc, AUG, (lambda t: sr.add_relative(t, prior, 30, ctx_m)) if rel else None)
    kw = dict(min_choice=3, wide_tau=0.95, override=None, **(dict(jetson=True, m=ctx_m) if ctx_m else {}))
    names = st.data + "_x" + st.scale_factor.astype(str)
    tau = cj.ltt_tau(cj.scan_tau(st.assign(data=names), p_oof, cj.TAUS, **kw), list(names.unique()), 0.005)
    tau = cj.NO_ANSWER if tau is None else max(tau, 0.9)
    del st, p_oof; gc.collect()
    ev = T if ctx_m is None else cj.add_context(T, m=ctx_m)
    ev = sr.add_relative(ev, prior, 30, ctx_m) if rel else ev
    return ev, sr.predict(model, ev, feats), tau, kw
VAR = {("pc", "V3 scale augmentation"): variant(F, {"li", "z", "hull", "cover9"}, False, None),
       ("pc", "V5 relative + augmentation"): variant(FR, {"rli", "z", "hull", "cover9"}, True, None),
       ("jetson", "V5 relative + augmentation"): variant(FR, {"rli", "z", "hull", "cover9"}, True, 5)}
{k: v[2] for k, v in VAR.items()}

In [ ]:
K = dict(min_choice=3, wide_tau=0.95)
RULES = {("pc", "bar 8 / 11"): lambda: cj.baseline(T, 8, 11, min_choice=3),
         ("pc", "judge 0.9 + override 12 (recommended)"): lambda: cj.decide(T, P, 0.9, override=12, **K),
         ("pc", "judge 0.9, no override"): lambda: cj.decide(T, P, 0.9, override=None, **K),
         ("pc", "judge 0.98, no override"): lambda: cj.decide(T, P, 0.98, override=None, min_choice=3, wide_tau=0.98),
         ("jetson", "Jetson today"): lambda: cj.baseline(T, 8, 8, jetson=True, m=5),
         ("jetson", "judge 0.925 + override 12 + early (recommended)"): lambda: cj.decide(T5, P5, 0.925, override=12, jetson=True, m=5, early_tau=0.98, **K),
         ("jetson", "judge 0.925, no override, early"): lambda: cj.decide(T5, P5, 0.925, override=None, jetson=True, m=5, early_tau=0.98, **K),
         ("jetson", "judge 0.98, no override, early"): lambda: cj.decide(T5, P5, 0.98, override=None, jetson=True, m=5, early_tau=0.98, min_choice=3, wide_tau=0.98)}
for (var, name), (ev, pv, tv, kwv) in VAR.items():
    RULES[(var, name)] = (lambda ev=ev, pv=pv, tv=tv, kwv=kwv: cj.decide(ev, pv, tv, **kwv))
rows, photos = [], []
for (var, name), f in RULES.items():
    r = f(); photos.append(r.assign(var=var, rule=name))
    s = cj.summary(r)
    for d in s.index:
        base, kind = d.split("_", 1)
        rows.append(dict(var=var, rule=name, data=base, alteration=kind, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), answered=s.loc[d, "answered"], queries=s.loc[d, "queries"]))
ALT = pd.DataFrame(rows); ALT.to_csv(RES / "it3_altered.csv", index=False)
pd.concat(photos).to_csv(RES / "it3_altered_photos.csv", index=False)
order = ["clean"] + KINDS
for d in ["official", "nudt707"]:
    print("==", d)
    a = ALT[ALT.data == d]
    display(pd.concat({"wrong": a.pivot_table(index=["var", "rule"], columns="alteration", values="wrong")[order].astype(int),
                       "score": a.pivot_table(index=["var", "rule"], columns="alteration", values="score")[order].round(1)}, axis=1))

In [ ]:
# total wrong answers per rule over the 14 altered runs (840 photo-runs), and the score kept relative to clean
tot = ALT[ALT.alteration != "clean"].groupby(["var", "rule"]).agg(wrong=("wrong", "sum"), mean_score=("score", "mean"))
cl = ALT[ALT.alteration == "clean"].groupby(["var", "rule"]).score.mean().rename("clean_score")
tot.join(cl).round(2)